Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os

print(os.path.exists("/content/drive/MyDrive/train_embeddings.npy"))
print(os.path.exists("/content/drive/MyDrive/train_meta.json"))

True
True


Install independencies

In [4]:
!pip install "git+https://github.com/huggingface/transformers.git" accelerate bitsandbytes qwen-vl-utils --break-system-packages

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-jm1onvzu
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-jm1onvzu
  Resolved https://github.com/huggingface/transformers.git to commit 9120f5e4858183b0b3fccf091869c026c530b898
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 31.8 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.7.0.dev0-py3-none-any.whl size=11612749 sha256=dad5ac567a30140393fbd2cd77e56fd288dfec478e0a4bec83677043206e5b10
  Stored in directory: /tmp/pip-ephem-wheel-cache-3vfnuv1x/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found ex

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print("显存总量:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
print("当前已用:", torch.cuda.memory_allocated() / 1e9, "GB")

Tesla T4
显存总量: 15.637086208 GB
当前已用: 0.0 GB


Load the model

In [5]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

model_name = "Qwen/Qwen3-VL-4B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading Qwen3-VL-4B...")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_name)
model.eval()
print("Qwen3-VL-4B loaded!")

Loading Qwen3-VL-4B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Qwen3-VL-4B loaded!


For one image

For multi images

删除错误结果

In [6]:
import os
import json
import re
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

# ===============================
# 加载 CLIP + 训练集 embeddings
# ===============================
print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

train_embeddings = np.load("/content/drive/MyDrive/train_embeddings.npy")
with open("/content/drive/MyDrive/train_meta.json", "r") as f:
    train_meta = json.load(f)

train_labels = train_meta["labels"]
train_filenames = train_meta["filenames"]
print(f"训练集 embeddings 加载完成，共 {len(train_embeddings)} 条")

# ===============================
# 路径配置
# ===============================
test_image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"
train_image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Train/Train_images"
output_json = "/content/drive/MyDrive/Qwen3VL4B_HM_FewShot_RAG_pred.json"

prompt_template = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Below are 3 reference examples with their correct labels retrieved from similar memes:

{examples}

Now classify the following meme:

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text, using the provided examples as reference to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

# ===============================
# RAG 检索函数
# ===============================
def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = clip_model.vision_model(**inputs)
        emb = outputs.pooler_output
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().numpy()

def retrieve_examples(test_emb):
    similarities = train_embeddings @ test_emb
    examples = {}
    for label in ["Homophobic", "Transphobic", "Non_Anti_LGBT"]:
        label_indices = [i for i, l in enumerate(train_labels) if l == label]
        label_sims = [(i, similarities[i]) for i in label_indices]
        label_sims.sort(key=lambda x: x[1], reverse=True)
        examples[label] = label_sims[0][0]
    return examples

def build_prompt(example_indices):
    label_map = {
        "Homophobic": "Homophobia",
        "Transphobic": "Transphobia",
        "Non_Anti_LGBT": "Non_LGBT"
    }
    examples_text = ""
    for i, (label, idx) in enumerate(example_indices.items()):
        examples_text += f"Example {i+1}: Class label: {label_map[label]}\n"
    return prompt_template.format(examples=examples_text)

# ===============================
# 获取测试图片列表
# ===============================
image_files = sorted(
    [f for f in os.listdir(test_image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png", ".gif"))],
    key=lambda x: int(re.search(r"(\d+)", x).group(1)) if re.search(r"(\d+)", x) else 0
)

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining = [f for f in image_files if f not in done_images]
print(f"剩余待处理: {len(remaining)} 张")

# ===============================
# 批量推理
# ===============================
for img_name in tqdm(remaining, desc="推理进度"):
    img_path = os.path.join(test_image_dir, img_name)

    try:
        torch.cuda.empty_cache()

        # RAG 检索
        test_emb = get_embedding(img_path)
        example_indices = retrieve_examples(test_emb)
        prompt_text = build_prompt(example_indices)

        # 加载图片并缩放
        image = Image.open(img_path).convert("RGB")
        if max(image.size) > 800:
            image.thumbnail((800, 800))

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt_text}
                ]
            }
        ]

        text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=text, images=image, return_tensors="pt").to(model.device)

        with torch.inference_mode():
            outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)

        gen = outputs[0][inputs["input_ids"].shape[-1]:]
        raw = processor.decode(gen, skip_special_tokens=True).strip()

        m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
        if m:
            label = m.group(1)
        elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry"]):
            label = "Non_LGBT"
        else:
            label = "UNKNOWN"

        # 统一大小写
        if label.lower() == "homophobia":
            label = "Homophobia"
        elif label.lower() == "transphobia":
            label = "Transphobia"
        elif label.lower() == "non_lgbt":
            label = "Non_LGBT"

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw
        })

        print(f"✅ {img_name} -> {label}")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        torch.cuda.empty_cache()

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e)
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        torch.cuda.empty_cache()

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

Loading CLIP...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

✅ 237.jpg -> Non_LGBT


推理进度: 100%|█████████▉| 231/232 [1:49:01<00:27, 27.54s/it]

✅ 238.jpg -> Non_LGBT


推理进度: 100%|██████████| 232/232 [1:49:28<00:00, 28.31s/it]

✅ 239.jpg -> Non_LGBT

完成！共 232 条结果已保存
  Homophobia: 40
  Non_LGBT: 183
  Transphobia: 7
  UNKNOWN: 2


In [7]:
import json

output_json = "/content/drive/MyDrive/Qwen3VL4B_HM_FewShot_RAG_pred.json"

with open(output_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

# 查看 UNKNOWN 的原始输出
unknowns = [p for p in predictions if p["predicted_label"] == "UNKNOWN"]
for u in unknowns:
    print(f"图片: {u['image_name']}")
    print(f"原始输出: {u['raw_output']}")
    print()

图片: 163.jpg
原始输出: Thought: The meme features a cartoon character with text that reads "拒绝搭讪" (refusing to engage in flirtatious or romantic advances) and "坚守t德" (upholding 't德' — a playful or coded reference to 't德' as a pun on 't德' meaning 't德' — which is a common internet slang for 't德' as a pun on 't德' meaning 't德' — which is a common internet slang for 't德' as a pun on 't德' meaning 't德' — which is a common internet slang for 't德' as a pun on 't德' meaning 't德' — which is a common internet slang for 't德' as a pun on 't德' meaning 't德' — which is a common internet slang for 't德' as a pun on 't德' meaning 't德' — which is a common internet slang for

图片: 184.jpg
原始输出: Thought: The meme depicts a confrontation between two characters, one of whom is identified as a "transgender" or "cross-gender" individual (represented as "我为跨性别发声" — "I speak for trans people"), and the other as a "straight male" (represented as "群体就是极端男权" — "the group is extreme male supremacy"). The text uses derogatory 

In [8]:
import json

output_json = "/content/drive/MyDrive/Qwen3VL4B_HM_FewShot_RAG_pred.json"

with open(output_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

for p in predictions:
    if p["image_name"] == "163.jpg":
        p["predicted_label"] = "Non_LGBT"  # 模型循环输出，内容无明显LGBT歧视
    if p["image_name"] == "184.jpg":
        p["predicted_label"] = "Transphobia"  # 原始输出明确描述了跨性别歧视内容

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

labels = [p["predicted_label"] for p in predictions]
print("最终统计:")
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

最终统计:
  Homophobia: 40
  Non_LGBT: 184
  Transphobia: 8


Check saved or not